<a href="https://colab.research.google.com/github/guebin/DL2026/blob/master/posts/10wk-1.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" style="text-align: left"></a>

# 1. 강의영상

{{<video https://youtu.be/playlist?list=PLQqh36zP38-zCxpUJ7HErPrSsG_gNCB80&si=H7eIf_GXnOZsi3Y3>}} 

# 2. Imports

In [1]:
import torch
import matplotlib.pyplot as plt

In [2]:
plt.rcParams['figure.figsize'] = (4.5,3.0)

# 3. CPU vs GPU

`-` 컴퓨터에 대한 상식

- 연산: CPU 혹은 GPU
- 값이저장되는곳: 시스템 RAM, GPU RAM

`-` 일반적으로는 시스템RAM에 값을 저장하고 CPU를 이용하여 연산하지만, 때로는 GPU RAM에 값을 저장하고 GPU를 이용하여 연산하는 경우도 있음.

`-` GPU가 사용가능한 상태인지 확인해야함. 그 명령어가 아래와 같음

In [ ]:
!nvidia-smi # GPU가 사용하지 않음

/bin/bash: line 1: nvidia-smi: command not found


`-` GPU를 할당받는 방법: "런타임 $\to$ 런타임 유형 변경 $\to$ T4 GPU"

In [ ]:
!nvidia-smi # GPU가 사용하지 않음

Sat May  9 20:23:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P8             15W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## A. CPU/GPU를 이용한 연산

`-` cpu를 이용한 연산

In [ ]:
Xcpu = torch.tensor([0.0,0.1,0.2]).reshape(-1,1)
net = torch.nn.Linear(1,1)

In [ ]:
net(Xcpu)

tensor([[-0.5513],
        [-0.5226],
        [-0.4938]], grad_fn=<AddmmBackward0>)

`-` gpu를 이용한 연산

In [ ]:
Xgpu = Xcpu.to("cuda:0")
Xgpu

tensor([[0.0000],
        [0.1000],
        [0.2000]], device='cuda:0')

In [ ]:
net(Xgpu) # 단순하게 아래와 같이 하면 error가 발생함

RuntimeError: Expected all tensors to be on the same device, but got mat1 is on cuda:0, different from other tensors on cpu (when checking argument in method wrapper_CUDA_addmm)

In [ ]:
net.weight, net.bias # 왜?? net.weight, net.bias가 모두 GPU RAM이 아니라 시스템 RAM에 저장되어있기 떄문

(Parameter containing:
 tensor([[0.2875]], requires_grad=True),
 Parameter containing:
 tensor([-0.5513], requires_grad=True))

In [ ]:
net.to("cuda:0") # 시스템 RAM에 있는 net의 파라메터들을 GPU RAM으로 옮기는 명령어

Linear(in_features=1, out_features=1, bias=True)

In [ ]:
net.weight, net.bias # 옮겨짐

(Parameter containing:
 tensor([[0.2875]], device='cuda:0', requires_grad=True),
 Parameter containing:
 tensor([-0.5513], device='cuda:0', requires_grad=True))

In [ ]:
net(Xgpu) # 이제 연산가능

tensor([[-0.5513],
        [-0.5226],
        [-0.4938]], device='cuda:0', grad_fn=<AddmmBackward0>)

`-` 정리하면, gpu는 gpu끼리 연산가능하고 cpu는 cpu까리 연산가능

(예시1)

In [ ]:
net.to("cpu")
net(Xcpu)

tensor([[-0.5513],
        [-0.5226],
        [-0.4938]], grad_fn=<AddmmBackward0>)

(예시2)

In [ ]:
net.to("cuda:0")
net(Xgpu)

tensor([[-0.5513],
        [-0.5226],
        [-0.4938]], device='cuda:0', grad_fn=<AddmmBackward0>)

(예시3)

In [ ]:
net.to("cpu")
net(Xgpu)

RuntimeError: Expected all tensors to be on the same device, but got mat1 is on cuda:0, different from other tensors on cpu (when checking argument in method wrapper_CUDA_addmm)

(예시4)

In [ ]:
net.to("cuda:0")
net(Xcpu)

RuntimeError: Expected all tensors to be on the same device, but got mat1 is on cpu, different from other tensors on cuda:0 (when checking argument in method wrapper_CUDA_addmm)

(예시5)

In [ ]:
net.to("cpu")
Ycpu = torch.tensor([0.0,0.2,0.4]).reshape(-1,1)
torch.mean((Ycpu-net(Xcpu))**2)

tensor(0.5416, grad_fn=<MeanBackward0>)

(예시6)

In [ ]:
net.to("cuda:0")
Ygpu = torch.tensor([0.0,0.2,0.4]).reshape(-1,1).to("cuda:0")
torch.mean((Ygpu-net(Xgpu))**2)

tensor(0.5416, device='cuda:0', grad_fn=<MeanBackward0>)

(예시7)

In [ ]:
net.to("cpu")
torch.mean((Ygpu-net(Xcpu))**2)

RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu!

(예시8)

In [ ]:
net.to("cuda:0")
torch.mean((Ycpu-net(Xgpu))**2)

RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu!

## B. 시간측정 (예비학습)

In [ ]:
import time

In [ ]:
t1 = time.time()

In [ ]:
t2 = time.time()

In [ ]:
t2-t1

3.3674073219299316

## C. CPU vs GPU

`-` CPU (512)

In [ ]:
X = torch.randn(100,1)
Y = torch.randn(100,1)
net = torch.nn.Sequential(
    torch.nn.Linear(1,512),
    torch.nn.ReLU(),
    torch.nn.Linear(512,1),
)
loss_fn = torch.nn.MSELoss()
optimizer = torch.optim.Adam(net.parameters())
#---#
t1 = time.time()
for epoch in range(1000):
    # 1
    Yhat = net(X)
    # 2
    loss = loss_fn(Yhat,Y)
    # 3
    loss.backward()
    # 4
    optimizer.step()
    optimizer.zero_grad()
t2 = time.time()
t2-t1

1.0356519222259521

`-` GPU (512)

In [ ]:
X = torch.randn(100,1).to("cuda:0")
Y = torch.randn(100,1).to("cuda:0")
net = torch.nn.Sequential(
    torch.nn.Linear(1,512),
    torch.nn.ReLU(),
    torch.nn.Linear(512,1),
).to("cuda:0")
loss_fn = torch.nn.MSELoss()
optimizer = torch.optim.Adam(net.parameters())
#---#
t1 = time.time()
for epoch in range(1000):
    # 1
    Yhat = net(X)
    # 2
    loss = loss_fn(Yhat,Y)
    # 3
    loss.backward()
    # 4
    optimizer.step()
    optimizer.zero_grad()
t2 = time.time()
t2-t1

1.468613862991333

`-` CPU (20480)

In [ ]:
X = torch.randn(100,1)
Y = torch.randn(100,1)
net = torch.nn.Sequential(
    torch.nn.Linear(1,20480),
    torch.nn.ReLU(),
    torch.nn.Linear(20480,1),
)
loss_fn = torch.nn.MSELoss()
optimizer = torch.optim.Adam(net.parameters())
#---#
t1 = time.time()
for epoch in range(1000):
    # 1
    Yhat = net(X)
    # 2
    loss = loss_fn(Yhat,Y)
    # 3
    loss.backward()
    # 4
    optimizer.step()
    optimizer.zero_grad()
t2 = time.time()
t2-t1

10.478125095367432

`-` GPU (20480)

In [ ]:
X = torch.randn(100,1).to("cuda:0")
Y = torch.randn(100,1).to("cuda:0")
net = torch.nn.Sequential(
    torch.nn.Linear(1,20480),
    torch.nn.ReLU(),
    torch.nn.Linear(20480,1),
).to("cuda:0")
loss_fn = torch.nn.MSELoss()
optimizer = torch.optim.Adam(net.parameters())
#---#
t1 = time.time()
for epoch in range(1000):
    # 1
    Yhat = net(X)
    # 2
    loss = loss_fn(Yhat,Y)
    # 3
    loss.backward()
    # 4
    optimizer.step()
    optimizer.zero_grad()
t2 = time.time()
t2-t1

1.4007525444030762

- 왜 이런차이가 나는가?
- CPU는 연산은 빠르지만 코어숫자가 적고, GPU는 연산은 느리지만 코어숫자가 많기 떄문

`-` CPU (204800)

In [ ]:
X = torch.randn(100,1)
Y = torch.randn(100,1)
net = torch.nn.Sequential(
    torch.nn.Linear(1,204800),
    torch.nn.ReLU(),
    torch.nn.Linear(204800,1),
)
loss_fn = torch.nn.MSELoss()
optimizer = torch.optim.Adam(net.parameters())
#---#
t1 = time.time()
for epoch in range(1000):
    # 1
    Yhat = net(X)
    # 2
    loss = loss_fn(Yhat,Y)
    # 3
    loss.backward()
    # 4
    optimizer.step()
    optimizer.zero_grad()
t2 = time.time()
t2-t1

248.22365045547485

In [ ]:
X = torch.randn(100,1).to("cuda:0")
Y = torch.randn(100,1).to("cuda:0")
net = torch.nn.Sequential(
    torch.nn.Linear(1,204800),
    torch.nn.ReLU(),
    torch.nn.Linear(204800,1),
).to("cuda:0")
loss_fn = torch.nn.MSELoss()
optimizer = torch.optim.Adam(net.parameters())
#---#
t1 = time.time()
for epoch in range(1000):
    # 1
    Yhat = net(X)
    # 2
    loss = loss_fn(Yhat,Y)
    # 3
    loss.backward()
    # 4
    optimizer.step()
    optimizer.zero_grad()
t2 = time.time()
t2-t1

4.633766412734985